# 🌍 Project 21 — Mini Environmental Monitoring Dashboard

## 🎯 Goal

Create a simple environmental monitoring dashboard that displays:

- 🛰️ Satellite imagery
- 🌿 NDVI
- 💧 NDWI
- 🏙️ Land Cover
- ⛰️ Elevation
- 📈 NDVI Time Series Chart

All in one Earth Engine script.

---

# 🌍 Why is this useful in Remote Sensing?

Environmental dashboards are used for:

- 🌍 Environmental monitoring
- 🌳 Forest management
- 🌾 Agriculture
- 💧 Water resource management
- 🔥 Disaster assessment
- 📊 Decision making

Instead of looking at one dataset, researchers combine multiple datasets to better understand an area.

---

# 🆕 New Skill: Integrating Multiple Datasets

Previously:

```text
One Dataset
      ↓
One Analysis
      ↓
One Result
```

Today:

```text
Satellite Image
        ↓

NDVI
        ↓

NDWI
        ↓

Land Cover
        ↓

Elevation
        ↓

Time Series
        ↓

Dashboard
```

You're combining many analyses into one project.

---

# 🔗 Connect to Previous Lessons

This project is a review of almost everything you've learned.

| Project | Skill |
|---------|-------|
| Project 1 | Satellite imagery |
| Project 6 | NDVI |
| Project 9 | NDWI |
| Project 12 | Land Cover |
| Project 15 | Elevation |
| Project 16 | NDVI Time Series Chart |

Nothing here is completely new.

The challenge is learning how to organize a larger Earth Engine script.

---

# 📊 Dashboard Workflow

```text
Study Area
      ↓
Load Satellite Image
      ↓
Create NDVI
      ↓
Create NDWI
      ↓
Load Land Cover
      ↓
Load DEM
      ↓
Create NDVI Chart
      ↓
Display Everything
```

---

# Step 1 — Define the Study Area

We'll use Germany again.

```javascript
var countryArea = ee.Geometry.Polygon([
  [
    [5.87,47.27],
    [15.04,47.27],
    [15.04,55.06],
    [5.87,55.06],
    [5.87,47.27]
  ]
]);
```

---

# Step 2 — Load a Sentinel-2 Image

```javascript
var image = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(countryArea)
.filterDate('2024-06-01','2024-06-30')
.sort('CLOUDY_PIXEL_PERCENTAGE')
.first()
.clip(countryArea);
```

This image will be used to calculate both **NDVI** and **NDWI**.

---

# Step 3 — Calculate NDVI

```javascript
var ndvi = image
  .normalizedDifference(['B8','B4'])
  .rename('NDVI');
```

---

# Step 4 — Calculate NDWI

```javascript
var ndwi = image
  .normalizedDifference(['B3','B8'])
  .rename('NDWI');
```

---

# Step 5 — Load Land Cover

```javascript
var landCover = ee.ImageCollection(
  "ESA/WorldCover/v200"
)
.first()
.select('Map')
.clip(countryArea);
```

---

# Step 6 — Load Elevation

```javascript
var dem = ee.Image(
  "USGS/SRTMGL1_003"
).clip(countryArea);
```

---

# Step 7 — Create an NDVI Time Series

Reuse the skills from **Project 16**.

```javascript
var collection = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(countryArea)
.filterDate('2024-01-01','2024-12-31');

function calculateNDVI(image){
  return image
    .normalizedDifference(['B8','B4'])
    .rename('NDVI');
}

var ndviCollection = collection.map(calculateNDVI);

var chart = ui.Chart.image.series({
  imageCollection: ndviCollection,
  region: countryArea,
  reducer: ee.Reducer.mean(),
  scale:10,
  xProperty:'system:time_start'
});

print(chart);
```

---

# Step 8 — Visualization Settings

```javascript
var trueColor = {
  bands:['B4','B3','B2'],
  min:0,
  max:3000
};

var ndviVis = {
  min:-1,
  max:1,
  palette:['brown','white','green']
};

var ndwiVis = {
  min:-1,
  max:1,
  palette:['brown','white','blue']
};

var landCoverVis = {
  min:10,
  max:100,
  palette:[
    '006400',
    'ffbb22',
    'ffff4c',
    'f096ff',
    'fa0000',
    '0064c8'
  ]
};

var demVis = {
  min:0,
  max:3000,
  palette:[
    'green',
    'yellow',
    'brown',
    'white'
  ]
};
```

---

# Step 9 — Display Everything

```javascript
Map.centerObject(countryArea,5);

Map.addLayer(
  image,
  trueColor,
  'Satellite Image'
);

Map.addLayer(
  ndvi,
  ndviVis,
  'NDVI'
);

Map.addLayer(
  ndwi,
  ndwiVis,
  'NDWI'
);

Map.addLayer(
  landCover,
  landCoverVis,
  'Land Cover'
);

Map.addLayer(
  dem,
  demVis,
  'Elevation'
);
```

Turn the layers on and off to compare them.

For example:

- Does dense vegetation (high NDVI) occur in forest areas?
- Do rivers identified by NDWI match the land cover map?
- How does elevation relate to vegetation?

---

# 💻 Complete Code

```javascript
// 1. Study Area
var countryArea = ee.Geometry.Polygon([
  [
    [5.87,47.27],
    [15.04,47.27],
    [15.04,55.06],
    [5.87,55.06],
    [5.87,47.27]
  ]
]);

// 2. Satellite Image
var image = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(countryArea)
.filterDate('2024-06-01','2024-06-30')
.sort('CLOUDY_PIXEL_PERCENTAGE')
.first()
.clip(countryArea);

// 3. NDVI
var ndvi = image
  .normalizedDifference(['B8','B4'])
  .rename('NDVI');

// 4. NDWI
var ndwi = image
  .normalizedDifference(['B3','B8'])
  .rename('NDWI');

// 5. Land Cover
var landCover = ee.ImageCollection(
  "ESA/WorldCover/v200"
)
.first()
.select('Map')
.clip(countryArea);

// 6. Elevation
var dem = ee.Image(
  "USGS/SRTMGL1_003"
).clip(countryArea);

// 7. NDVI Time Series
var collection = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(countryArea)
.filterDate('2024-01-01','2024-12-31');

function calculateNDVI(image){
  return image
    .normalizedDifference(['B8','B4'])
    .rename('NDVI');
}

var ndviCollection = collection.map(calculateNDVI);

var chart = ui.Chart.image.series({
  imageCollection: ndviCollection,
  region: countryArea,
  reducer: ee.Reducer.mean(),
  scale:10,
  xProperty:'system:time_start'
});

print(chart);

// 8. Visualization
var trueColor = {
  bands:['B4','B3','B2'],
  min:0,
  max:3000
};

var ndviVis = {
  min:-1,
  max:1,
  palette:['brown','white','green']
};

var ndwiVis = {
  min:-1,
  max:1,
  palette:['brown','white','blue']
};

var landCoverVis = {
  min:10,
  max:100,
  palette:[
    '006400',
    'ffbb22',
    'ffff4c',
    'f096ff',
    'fa0000',
    '0064c8'
  ]
};

var demVis = {
  min:0,
  max:3000,
  palette:['green','yellow','brown','white']
};

// 9. Display
Map.centerObject(countryArea,5);

Map.addLayer(image, trueColor, 'Satellite Image');
Map.addLayer(ndvi, ndviVis, 'NDVI');
Map.addLayer(ndwi, ndwiVis, 'NDWI');
Map.addLayer(landCover, landCoverVis, 'Land Cover');
Map.addLayer(dem, demVis, 'Elevation');
```

---

# 📌 Things to Remember

## 1. Environmental Monitoring Uses Multiple Datasets

Different datasets answer different questions.

| Dataset | Purpose |
|---------|---------|
| Satellite Image | What does the area look like? |
| NDVI | How healthy is the vegetation? |
| NDWI | Where is the water? |
| Land Cover | What type of land is present? |
| DEM | What is the elevation? |
| NDVI Time Series Chart | How does vegetation change over time? |

---

## 2. Organize Large Scripts

As your projects grow, group your code into clear sections:

```text
Study Area
      ↓
Load Data
      ↓
Process Data
      ↓
Visualization
      ↓
Display
```

This makes your code easier to read, debug, and maintain.

---

## 3. Think Like a Researcher

Instead of looking at one map, compare multiple layers to answer scientific questions.

For example:

```text
High NDVI
      +
Forest Land Cover
      +
High Elevation
      ↓
Understand Vegetation Patterns
```

By combining different datasets, you can reveal relationships that a single dataset cannot show.

---

# ❌ Common Beginner Mistakes

### ❌ Mistake 1: Using Different Study Areas

Always clip **all datasets** to the **same study area** so they can be compared directly.

---

### ❌ Mistake 2: Using Different Dates

For example, calculating NDVI from **June** but comparing it with a satellite image from **December** may lead to misleading conclusions.

Whenever possible, use images from the same time period for related analyses.

---

### ❌ Mistake 3: Forgetting the Purpose of Each Layer

Remember that each dataset provides different information.

- Satellite imagery shows what the Earth looks like.
- NDVI measures vegetation health.
- NDWI highlights water.
- Land Cover classifies surface types.
- DEM represents elevation.
- The NDVI chart shows how vegetation changes over time.

Don't expect every layer to show the same patterns.

---

# 💻 Your Practice (Project 21.1)

Create your own environmental monitoring dashboard for a study area of your choice.

## Requirements

- Create a study area.
- Load a Sentinel-2 image.
- Calculate NDVI.
- Calculate NDWI.
- Load the ESA WorldCover dataset.
- Load the SRTM DEM.
- Create an NDVI time series chart.
- Display all layers on the map.
- Compare the different datasets.

---

# 🧠 Before Writing the Code

## Question 1

Why is it useful to combine **multiple datasets** (such as NDVI, NDWI, Land Cover, and Elevation) instead of relying on only one dataset?

---

## Question 2

Suppose you observe:

- 🌿 High NDVI
- 🏙️ Built-up land cover
- ⛰️ High elevation

What questions or hypotheses could you investigate by comparing these layers together?